In [9]:
import os
import sys
import pandas as pd
import numpy as np

# Add the project root to Python's path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.load_data import load_data

df = load_data()

print(df.shape)

(1067371, 8)


In [3]:
import os
import sys

# Add the project root to Python's path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.load_data import load_data

df = load_data()

print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
# Find the last transaction date in the dataset
snapshot_date = df["InvoiceDate"].max()

print("Last transaction date:", snapshot_date)

# Customer's last purchase date
last_purchase = (
    df.groupby("Customer ID")["InvoiceDate"]
      .max()
      .reset_index()
)

# Calculate Recency (days since last purchase)
last_purchase["Recency"] = (
    snapshot_date - last_purchase["InvoiceDate"]
).dt.days

# Create Churn column
last_purchase["Churn"] = (
    last_purchase["Recency"] > 90
).astype(int)

# Display first few rows
last_purchase.head()

Last transaction date: 2011-12-09 12:50:00


,Customer ID,InvoiceDate,Recency,Churn
0,12346.0,2011-01-18 10:17:00,325,1
1,12347.0,2011-12-07 15:52:00,1,0
2,12348.0,2011-09-25 13:13:00,74,0
3,12349.0,2011-11-21 09:51:00,18,0
4,12350.0,2011-02-02 16:01:00,309,1


In [2]:
import os

print(os.getcwd())

c:\Users\ADMIN\OneDrive\Documents\GitHub\Customer-behavior-prediction\notebooks


In [5]:
# Load customer-level features
customer_features = pd.read_csv("../data/processed/customer_features.csv")

# Merge with churn labels
model_data = customer_features.merge(
    last_purchase[["Customer ID", "Churn"]],
    on="Customer ID",
    how="left"
)

# Check the result
print(model_data.shape)
model_data.head()

NameError: name 'pd' is not defined

In [10]:
customer_features = pd.read_csv("../data/processed/customer_features.csv")

print(customer_features.shape)
customer_features.head()

(5942, 7)


,Customer ID,TotalOrders,TotalProducts,AverageOrderValue,Recency,Frequency,Monetary
0,12346.0,17,52,-1.347500,326,17,-64.68
1,12347.0,8,3286,22.266087,2,8,5633.32
2,12348.0,5,2714,39.596078,75,5,2019.40
3,12349.0,5,1619,24.469667,19,5,4404.54
4,12350.0,1,197,19.670588,310,1,334.40


In [11]:
# Merge customer features with churn labels
model_data = customer_features.merge(
    last_purchase[["Customer ID", "Churn"]],
    on="Customer ID",
    how="left"
)

# Check the merged dataset
print(model_data.shape)
model_data.head()

(5942, 8)


,Customer ID,TotalOrders,TotalProducts,AverageOrderValue,Recency,Frequency,Monetary,Churn
0,12346.0,17,52,-1.347500,326,17,-64.68,1
1,12347.0,8,3286,22.266087,2,8,5633.32,0
2,12348.0,5,2714,39.596078,75,5,2019.40,0
3,12349.0,5,1619,24.469667,19,5,4404.54,0
4,12350.0,1,197,19.670588,310,1,334.40,1


In [12]:
# Check class balance
print(model_data["Churn"].value_counts())

print("\nPercentage:")

print(model_data["Churn"].value_counts(normalize=True) * 100)

Churn
1    3020
0    2922
Name: count, dtype: int64

Percentage:
Churn
1    50.824638
0    49.175362
Name: proportion, dtype: float64


In [13]:
from sklearn.model_selection import train_test_split

# Features (X)
X = model_data.drop(columns=["Customer ID", "Churn"])

# Target (y)
y = model_data["Churn"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

Training set: (4753, 6)
Testing set : (1189, 6)


In [14]:
from sklearn.linear_model import LogisticRegression

# Create the model
log_model = LogisticRegression(max_iter=1000)

# Train the model
log_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

# Evaluation Metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.9983179142136249
Precision: 1.0
Recall   : 0.9966887417218543
F1 Score : 0.9983416252072969
ROC-AUC  : 0.9999886794588781

Confusion Matrix
[[585   0]
 [  2 602]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       585
           1       1.00      1.00      1.00       604

    accuracy                           1.00      1189
   macro avg       1.00      1.00      1.00      1189
weighted avg       1.00      1.00      1.00      1189

